In [16]:
%pip install -q transformers langchain langchain-community langchain-text-splitters langchain-huggingface

In [17]:
%pip install -q torch sentence-transformers faiss-cpu pypdf

In [18]:
%pip install llama-cpp-python

In [43]:

%pip install -q "huggingface-hub>=0.34.0,<1.0"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from langchain_community.llms import LlamaCpp

print("LlamaCpp is ready!")

LlamaCpp is ready!


/tmp/ipykernel_22993/764687592.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import LlamaCpp


In [3]:
import os

# Load PDF documents
from langchain_community.document_loaders import PyPDFLoader

# Split documents into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create text embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Store and search vector embeddings
from langchain_community.vectorstores import FAISS

# Load FLAN-T5 and TinyLlama using Transformers
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

# Load TinyLlama GGUF using llama.cpp
from langchain_community.llms import LlamaCpp

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os
import zipfile

ZIP_PATH = "/content/drive/MyDrive/Gen_AI_Dataset/PrepMate_AI_Interview_PDFs.zip"
BOOKS_PATH = "/content/drive/MyDrive/Gen_AI_Dataset/books"

# Create the books folder
os.makedirs(BOOKS_PATH, exist_ok=True)

# Extract the ZIP into books
with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(BOOKS_PATH)

# Check the extracted files
print("Books folder created:", os.path.isdir(BOOKS_PATH))
print("Files inside books:")

for root, dirs, files in os.walk(BOOKS_PATH):
    for file in files:
        print(os.path.join(root, file))

Books folder created: True
Files inside books:
/content/drive/MyDrive/Gen_AI_Dataset/books/Python_Interview.pdf
/content/drive/MyDrive/Gen_AI_Dataset/books/NLP_Interview.pdf
/content/drive/MyDrive/Gen_AI_Dataset/books/ML_Interview.pdf
/content/drive/MyDrive/Gen_AI_Dataset/books/DL_Interview.pdf


In [6]:
BOOKS_PATH = "/content/drive/MyDrive/Gen_AI_Dataset/books"

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader

pdf_files = [
    os.path.join(BOOKS_PATH, file)
    for file in os.listdir(BOOKS_PATH)
    if file.lower().endswith(".pdf")
]

documents = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    pages = loader.load()
    documents.extend(pages)

print("PDF files loaded:", len(pdf_files))
print("Total pages loaded:", len(documents))

PDF files loaded: 4
Total pages loaded: 16


### **Split Documents into Text Chunks**

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 47


### **Create Text Embeddings and Build the FAISS Vector Index**

In [9]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Embeddings created and FAISS index built successfully!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Embeddings created and FAISS index built successfully!


In [10]:
# Save the FAISS vector index to Google Drive
FAISS_PATH = "/content/drive/MyDrive/Gen_AI_Dataset/faiss_index"

vectorstore.save_local(FAISS_PATH)

print("FAISS index saved successfully!")

FAISS index saved successfully!


In [11]:
##2. Create the retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print("Retriever created successfully!")

Retriever created successfully!


In [12]:
query = "What is machine learning?"

results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)


--- Result 1 ---
PrepMate AI | Original Interview Preparation Guide
Page 2
Foundations
1. What is machine learning?
Machine learning uses data to learn patterns that support predictions or decisions without manually programming
every rule. A typical workflow includes defining the target, collecting data, preparing features, training, evaluating,
and monitoring a model.
2. What is the difference between supervised and unsupervised learning?
Supervised learning uses labeled examples, such as inputs paired with known outcomes. Unsupervised learning
looks for structure in data without target labels, such as clusters or lower-dimensional representations.
3. Explain classification and regression.
Classification predicts a category, such as fraud or not fraud. Regression predicts a numerical value, such as

--- Result 2 ---
interpretable and often trained by minimizing squared error, subject to the model assumptions and chosen
objective.
9. What is logistic regression?
Logistic regression is

Load FLAN-T5


In [13]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

hf_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=150,
    do_sample=False
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

print("FLAN-T5 loaded successfully!")

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


FLAN-T5 loaded successfully!


In [14]:
question = "What is machine learning?"

# Retrieve relevant PDF chunks
docs = retriever.invoke(question)
context = "\n\n".join(doc.page_content for doc in docs)

# Create the prompt
prompt = f"""Answer the question using the context below.
Give a simple, beginner-friendly answer.

Context:
{context[:1200]}

Question: {question}
Answer:"""

# Generate the FLAN-T5 answer
flan_answer = llm.invoke(prompt)

print("Question:", question)
print("FLAN-T5 Answer:", flan_answer)

Question: What is machine learning?
FLAN-T5 Answer: Machine learning uses data to learn patterns that support predictions or decisions without manually programming every rule.


# Load TinyLlama Hugging Face

In [29]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

tiny_pipeline = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=100,
    do_sample=False,
    return_full_text=False
)

tiny_llm = HuggingFacePipeline(
    pipeline=tiny_pipeline
)

print("TinyLlama loaded successfully!")

Device set to use cpu


TinyLlama loaded successfully!


In [31]:
question = "What is machine learning?"

docs = retriever.invoke(question)
context = "\n\n".join(doc.page_content for doc in docs)

prompt = f"""<|system|>
You are PrepMate AI, an interview preparation assistant.
Answer clearly using the provided context.
If the answer is not in the context, say you cannot find it.
</s>
<|user|>
Context:
{context[:1200]}

Question: {question}
</s>
<|assistant|>
"""

tiny_answer = tiny_llm.invoke(prompt)

print("Question:", question)
print("TinyLlama Answer:", tiny_answer)

Question: What is machine learning?
TinyLlama Answer: Answer:
Machine learning is a field of artificial intelligence that uses data to learn patterns and make predictions without requiring human intervention. It involves the use of algorithms, statistical models, and machine learning techniques to analyze and interpret data. The foundations of machine learning include supervised and unsupervised learning, classification and regression, logistic regression, and decision trees.


In [17]:
question = "What is machine learning?"

print("QUESTION:", question)

print("\n--- FLAN-T5 ---")
print(flan_answer)

print("\n--- TinyLlama Hugging Face ---")
print(tiny_answer)

QUESTION: What is machine learning?

--- FLAN-T5 ---
Machine learning uses data to learn patterns that support predictions or decisions without manually programming every rule.

--- TinyLlama Hugging Face ---
Answer:
Machine learning is a field of artificial intelligence that uses data to learn patterns and make predictions without requiring human intervention. It involves the use of algorithms, statistical models, and machine learning techniques to analyze and interpret data. The foundations of machine learning include supervised and unsupervised learning, classification and regression, logistic regression, and decision trees.


In [32]:
#Check whether your GGUF model file exists
from huggingface_hub import hf_hub_download
import os

GGUF_MODEL_PATH = hf_hub_download(
    repo_id="TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
    filename="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
    local_dir="/content/drive/MyDrive/Gen_AI_Dataset"
)

print("GGUF model downloaded:", os.path.isfile(GGUF_MODEL_PATH))
print("Model path:", GGUF_MODEL_PATH)

GGUF model downloaded: True
Model path: /content/drive/MyDrive/Gen_AI_Dataset/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf


In [33]:
from langchain_community.llms import LlamaCpp

GGUF_MODEL_PATH = "/content/drive/MyDrive/Gen_AI_Dataset/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf"

gguf_llm = LlamaCpp(
    model_path=GGUF_MODEL_PATH,
    n_ctx=2048,
    max_tokens=100,
    temperature=0.1,
    verbose=False
)

print("TinyLlama GGUF loaded successfully!")

TinyLlama GGUF loaded successfully!


In [34]:
question = "What is machine learning?"

# Retrieve context from your PDFs
docs = retriever.invoke(question)
context = "\n\n".join(doc.page_content for doc in docs)

# Create prompt for TinyLlama GGUF
gguf_prompt = f"""<|system|>
You are PrepMate AI, an interview preparation assistant.
Answer clearly using the provided context.
If the answer is not in the context, say you cannot find it.
</s>
<|user|>
Context:
{context[:1200]}

Question: {question}
</s>
<|assistant|>
"""

# Generate answer
gguf_answer = gguf_llm.invoke(gguf_prompt)

print("Question:", question)
print("TinyLlama GGUF Answer:", gguf_answer)

Question: What is machine learning?
TinyLlama GGUF Answer: Machine learning is a field of artificial intelligence that involves the use of algorithms and data to learn patterns and make predictions without requiring human intervention. It can be broken down into two main categories: supervised and unsupervised learning. Supervised learning uses labeled examples, such as inputs paired with known outcomes, while unsupervised learning looks for structure in data without target labels, such as clusters or lower-dimensional representations. Classification and regression are the most common types of machine learning models used


### single-question evaluation

In [35]:
single_reference_answer = """
Machine learning is a branch of artificial intelligence
that enables computers to learn patterns from data and
make predictions or decisions.
"""

In [36]:

import re

def evaluate_with_judge(question, answer, context, reference_answer):

    correctness_prompt = f"""
Rate the factual correctness of the answer from 1 to 5.
5 = Fully correct
4 = Mostly correct
3 = Partially correct
2 = Mostly incorrect
1 = Incorrect

Question: {question}
Reference answer: {reference_answer}
Generated answer: {answer}

Return only one number from 1 to 5.
"""

    correctness_result = llm.invoke(correctness_prompt)

    groundedness_prompt = f"""
Rate how well the answer is supported by the context from 1 to 5.
5 = All claims are supported
4 = Most claims are supported
3 = Some claims are unsupported
2 = Most claims are unsupported
1 = Not supported

Context: {context[:1000]}
Question: {question}
Answer: {answer}

Return only one number from 1 to 5.
"""

    groundedness_result = llm.invoke(groundedness_prompt)

    def extract_score(result):
        match = re.search(
            r"\b([1-5])\b",
            str(result).strip()
        )
        return int(match.group(1)) if match else None

    return {
        "Correctness": extract_score(correctness_result),
        "Groundedness": extract_score(groundedness_result),
        "Correctness raw output": correctness_result,
        "Groundedness raw output": groundedness_result
    }

print("Evaluation function defined successfully!")


Evaluation function defined successfully!


In [37]:

# Define the question
question = "What is machine learning?"

# Retrieve relevant information from your PDFs
docs = retriever.invoke(question)
context = "\n\n".join(doc.page_content for doc in docs)

# Generate an answer using FLAN-T5
prompt = f"""
Answer the question using the provided context.
If the answer is not available, say so.

Context:
{context[:1000]}

Question: {question}
Answer:
"""

answer = llm.invoke(prompt)

# Evaluate the generated answer
scores = evaluate_with_judge(
    question=question,
    answer=answer,
    context=context,
    reference_answer=single_reference_answer
)

# Display the results
print("Question:", question)
print("\nReference Answer:", single_reference_answer)
print("\nGenerated Answer:", answer)
print("\nCorrectness Score:", scores["Correctness"], "/ 5")
print("Groundedness Score:", scores["Groundedness"], "/ 5")


Question: What is machine learning?

Reference Answer: 
Machine learning is a branch of artificial intelligence
that enables computers to learn patterns from data and
make predictions or decisions.


Generated Answer: Machine learning uses data to learn patterns that support predictions or decisions without manually programming every rule.

Correctness Score: 5 / 5
Groundedness Score: 5 / 5


In [38]:

reference_answers = {
    "What is machine learning?": """
    Machine learning is a branch of artificial intelligence
    that enables computers to learn patterns from data and
    make predictions or decisions.
    """,

    "What is supervised learning?": """
    Supervised learning is a type of machine learning in
    which a model learns from labeled data containing
    inputs and their correct outputs.
    """,

    "What is overfitting in machine learning?": """
    Overfitting occurs when a model learns the training
    data too closely, including noise, and performs poorly
    on new, unseen data.
    """,

    "What is a neural network?": """
    A neural network is a machine learning model made of
    interconnected artificial neurons arranged in layers
    that learn patterns from data.
    """,

    "What is the difference between AI and ML?": """
    Artificial intelligence is the broader field of
    creating systems that perform tasks requiring
    intelligence. Machine learning is a subset of AI
    that learns patterns from data.
    """
}

print("Five reference answers defined successfully!")


Five reference answers defined successfully!


In [40]:

import pandas as pd

all_results = []

for question, reference_answer in reference_answers.items():

    # 1. Retrieve relevant PDF chunks
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    # 2. FLAN-T5 prompt
    flan_prompt = f"""Answer the question using the context below.
Give a simple, beginner-friendly answer.
If the answer is not in the context, say you cannot find it.

Context:
{context[:1200]}

Question: {question}
Answer:"""

    # 3. TinyLlama Hugging Face prompt
    tiny_prompt = f"""<|system|>
You are PrepMate AI, an interview preparation assistant.
Answer clearly using the provided context.
If the answer is not in the context, say you cannot find it.
</s>
<|user|>
Context:
{context[:1200]}

Question: {question}
</s>
<|assistant|>
"""

    # 4. Generate answers using your existing models

    # FLAN-T5
    flan_answer = llm.invoke(flan_prompt)

    # TinyLlama Hugging Face
    tiny_answer = tiny_llm.invoke(tiny_prompt)

    # TinyLlama GGUF
    gguf_answer = gguf_llm.invoke(tiny_prompt)

    # 5. Collect answers from all models
    model_answers = {
        "FLAN-T5": flan_answer,
        "TinyLlama Hugging Face": tiny_answer,
        "TinyLlama GGUF": gguf_answer
    }

    # 6. Compare each generated answer with its reference answer
    for model_name, answer in model_answers.items():

        scores = evaluate_with_judge(
            question=question,
            answer=answer,
            context=context,
            reference_answer=reference_answer
        )

        all_results.append({
            "Question": question,
            "Model": model_name,
            "Reference Answer": reference_answer.strip(),
            "Generated Answer": answer,
            "Correctness": scores["Correctness"],
            "Groundedness": scores["Groundedness"]
        })

        print("\n" + "=" * 60)
        print("Question:", question)
        print("Model:", model_name)
        print("Reference Answer:", reference_answer.strip())
        print("Generated Answer:", answer)
        print("Correctness:", scores["Correctness"], "/ 5")
        print("Groundedness:", scores["Groundedness"], "/ 5")

# 7. Display all 15 evaluations
results_df = pd.DataFrame(all_results)

print("\nAll Model Results: 5 Questions x 3 Models")

display(
    results_df[
        [
            "Question",
            "Model",
            "Reference Answer",
            "Generated Answer",
            "Correctness",
            "Groundedness"
        ]
    ]
)

# 8. Calculate average scores for each model
print("\nAverage Scores by Model")

average_scores = (
    results_df
    .groupby("Model")[["Correctness", "Groundedness"]]
    .mean()
    .round(2)
)

display(average_scores)



Question: What is machine learning?
Model: FLAN-T5
Reference Answer: Machine learning is a branch of artificial intelligence
    that enables computers to learn patterns from data and
    make predictions or decisions.
Generated Answer: Machine learning uses data to learn patterns that support predictions or decisions without manually programming every rule.
Correctness: 5 / 5
Groundedness: 5 / 5

Question: What is machine learning?
Model: TinyLlama Hugging Face
Reference Answer: Machine learning is a branch of artificial intelligence
    that enables computers to learn patterns from data and
    make predictions or decisions.
Generated Answer: Answer:
Machine learning is a field of artificial intelligence that uses data to learn patterns and make predictions without requiring human intervention. It involves the use of algorithms, statistical models, and machine learning techniques to analyze and interpret data. The foundations of machine learning include supervised and unsupervised l

,Question,Model,Reference Answer,Generated Answer,Correctness,Groundedness
0,What is machine learning?,FLAN-T5,Machine learning is a branch of artificial int...,Machine learning uses data to learn patterns t...,5,5
1,What is machine learning?,TinyLlama Hugging Face,Machine learning is a branch of artificial int...,Answer:\nMachine learning is a field of artifi...,5,5
2,What is machine learning?,TinyLlama GGUF,Machine learning is a branch of artificial int...,Machine learning is a field of artificial inte...,5,1
3,What is supervised learning?,FLAN-T5,Supervised learning is a type of machine learn...,"Machine learning uses labeled examples, such a...",5,2
4,What is supervised learning?,TinyLlama Hugging Face,Supervised learning is a type of machine learn...,"Supervised learning uses labeled examples, suc...",5,1
5,What is supervised learning?,TinyLlama GGUF,Supervised learning is a type of machine learn...,Supervised learning is a type of machine learn...,5,1
6,What is overfitting in machine learning?,FLAN-T5,Overfitting occurs when a model learns the tra...,when a model learns training-specific patterns...,5,5
7,What is overfitting in machine learning?,TinyLlama Hugging Face,Overfitting occurs when a model learns the tra...,Overfitting occurs when a model learns trainin...,5,5
8,What is overfitting in machine learning?,TinyLlama GGUF,Overfitting occurs when a model learns the tra...,Overfitting in machine learning refers to the ...,5,5
9,What is a neural network?,FLAN-T5,A neural network is a machine learning model m...,Deep learning is a branch of machine learning ...,5,1



Average Scores by Model


,Correctness,Groundedness
Model,,
FLAN-T5,5.0,3.6
TinyLlama GGUF,5.0,1.8
TinyLlama Hugging Face,5.0,3.4


# Final RAG Pipeline: Generate Interview Answers Using FLAN-T5 and Retrieved PDF Context

In [42]:
def prepmate_answer(question):
    # 1. Retrieve relevant PDF chunks
    docs = retriever.invoke(question)

    if not docs:
        return "I could not find relevant information in the provided PDFs."

    # 2. Combine the retrieved chunks into context
    context = "\n\n".join(doc.page_content for doc in docs)
    context = context[:1200]

    # 3. Create the FLAN-T5 prompt
    prompt = f"""Answer the question clearly and simply
using only the provided context.

Do not invent facts. If the context does not contain
the answer, say you could not find it in the provided PDFs.

Context:
{context}

Question: {question}

Answer:"""

    # 4. Generate the answer using FLAN-T5
    answer = llm.invoke(prompt)

    return answer.strip()


# Test the final PrepMate AI model
question = "What is machine learning?"

final_answer = prepmate_answer(question)

print("Question:", question)
print("PrepMate AI Answer:", final_answer)

Question: What is machine learning?
PrepMate AI Answer: Machine learning uses data to learn patterns that support predictions or decisions without manually programming every rule.
